# NYC Taxi Analytics

**Dataset:** `samples.nyctaxi.trips`

**Difficulty:** Medium

**Topics:** derived columns, window functions, hour extraction, aggregation

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

trips = spark.read.table("samples.nyctaxi.trips")

## Problem 1

Calculate the trip duration in minutes for each trip (dropoff time minus pickup time).
Filter out trips where the duration is 0 or negative.

**Expected output columns:**
- `tpep_pickup_datetime`
- `tpep_dropoff_datetime`
- `trip_distance`
- `fare_amount`
- `duration_minutes`

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = trips.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    F.ceil((
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    )/60).cast("int").alias("duration_minutes")
).filter(F.col("duration_minutes") > 0)
result_1.display()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'tpep_pickup_datetime' in cols, "Missing column: tpep_pickup_datetime"
assert 'tpep_dropoff_datetime' in cols, "Missing column: tpep_dropoff_datetime"
assert 'trip_distance' in cols, "Missing column: trip_distance"
assert 'fare_amount' in cols, "Missing column: fare_amount"
assert 'duration_minutes' in cols, "Missing column: duration_minutes"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_dur = result_1.agg(F.min('duration_minutes')).collect()[0][0]
assert min_dur > 0, f"Expected all duration_minutes > 0, found min={min_dur}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Compute the total number of trips and total revenue grouped by the hour of day.
Extract the hour from `tpep_pickup_datetime`. Sort results by `pickup_hour` ascending.

**Expected output columns:**
- `pickup_hour`
- `trip_count`
- `total_revenue`

In [0]:
trips.printSchema()

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = trips.groupBy(F.hour("tpep_pickup_datetime").alias("pickup_hour")).agg(
    F.count("*").alias("trip_count"),
    F.sum("fare_amount").alias("total_revenue")
).orderBy("pickup_hour")
result_2.display()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'pickup_hour' in cols, "Missing column: pickup_hour"
assert 'trip_count' in cols, "Missing column: trip_count"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 24, f"Expected at most 24 hours, got {cnt} rows"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Calculate the fare per mile ratio for each trip (`fare_amount / trip_distance`).
Filter to only include trips where `trip_distance > 0`.

**Expected output columns:**
- `pickup_zip`
- `dropoff_zip`
- `trip_distance`
- `fare_amount`
- `fare_per_mile`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = trips.filter(F.col("trip_distance") > 0).select(
    "pickup_zip",
    "dropoff_zip",
    "trip_distance",
    "fare_amount",
    F.expr("fare_amount / trip_distance").alias("fare_per_mile")
)
result_3.display()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'dropoff_zip' in cols, "Missing column: dropoff_zip"
assert 'trip_distance' in cols, "Missing column: trip_distance"
assert 'fare_amount' in cols, "Missing column: fare_amount"
assert 'fare_per_mile' in cols, "Missing column: fare_per_mile"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_dist = result_3.agg(F.min('trip_distance')).collect()[0][0]
assert min_dist > 0, f"Expected trip_distance > 0 for all rows, found min={min_dist}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Find the top 10 pickup-dropoff zip code pairs by number of trips.
Sort by `trip_count` descending.

**Expected output columns:**
- `pickup_zip`
- `dropoff_zip`
- `trip_count`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = trips.groupBy("pickup_zip", "dropoff_zip").agg(
        F.count("*").alias("trip_count")
    ).orderBy(F.col("trip_count").desc()).limit(10)

result_4.display()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'dropoff_zip' in cols, "Missing column: dropoff_zip"
assert 'trip_count' in cols, "Missing column: trip_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 10, f"Expected at most 10 rows (top 10), got {cnt}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Using a window function, rank pickup zip codes by trip count within each dropoff zip.
Keep only rows where rank <= 3.

**Expected output columns:**
- `dropoff_zip`
- `pickup_zip`
- `trip_count`
- `rank`

In [0]:
trips.printSchema()

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = trips.groupBy(
    "dropoff_zip",
    "pickup_zip"
).agg(F.count("*").alias("trip_count")).withColumn(
    "rank",
    F.rank().over(Window.partitionBy("dropoff_zip").orderBy(F.col("trip_count").desc()))
).filter(F.col("rank") <= 3)

result_5.display()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'dropoff_zip' in cols, "Missing column: dropoff_zip"
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'trip_count' in cols, "Missing column: trip_count"
assert 'rank' in cols, "Missing column: rank"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rank = result_5.agg(F.max('rank')).collect()[0][0]
assert max_rank <= 3, f"Expected rank <= 3, found max rank={max_rank}"
min_rank = result_5.agg(F.min('rank')).collect()[0][0]
assert min_rank >= 1, f"Expected rank >= 1, found min rank={min_rank}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Compute average fare, average distance, and trip count grouped by `pickup_zip`.
Only include zip codes with more than 100 trips.

**Expected output columns:**
- `pickup_zip`
- `trip_count`
- `avg_fare`
- `avg_distance`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6

result_6 = trips.groupBy("pickup_zip").agg(
    F.count("*").alias("trip_count"),
    F.avg("fare_amount").alias("avg_fare"),
    F.avg("trip_distance").alias("avg_distance")
).filter(F.col("trip_count")>100)

In [0]:
result_6.display()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'trip_count' in cols, "Missing column: trip_count"
assert 'avg_fare' in cols, "Missing column: avg_fare"
assert 'avg_distance' in cols, "Missing column: avg_distance"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_trips = result_6.agg(F.min('trip_count')).collect()[0][0]
assert min_trips > 100, f"Expected all trip_count > 100, found min={min_trips}"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Calculate a running total of `fare_amount` ordered by `tpep_pickup_datetime` using a window function.

**Expected output columns:**
- `tpep_pickup_datetime`
- `fare_amount`
- `running_total_fare`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7

result_7 = trips.select(
    "tpep_pickup_datetime",
    "fare_amount",
    F.sum("fare_amount").over(Window.orderBy("tpep_pickup_datetime")).alias("running_total_fare")
)
result_7.display()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'tpep_pickup_datetime' in cols, "Missing column: tpep_pickup_datetime"
assert 'fare_amount' in cols, "Missing column: fare_amount"
assert 'running_total_fare' in cols, "Missing column: running_total_fare"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_running = result_7.agg(F.max('running_total_fare')).collect()[0][0]
assert max_running > 0, f"Expected running_total_fare > 0, got max={max_running}"
print(f"Problem 7 passed ✓  ({cnt} rows)")